# Activity: Can We Trust the AI Analyst? — FreshKart Case

**Unit 4 · AI-Assisted Visualization · LLMs for EDA & Human-in-the-Loop Visualization**
Visualization of Data, Understanding and Interpretation (UE24CS342AA9)

---

## The case

**FreshKart** is a quick-commerce grocery chain with **10 dark stores in 3 cities** (Bengaluru, Pune, Hyderabad). Operations head **Meera** has a leadership meeting tomorrow. She asked an AI analytics assistant:

> *"Summarise our business from January to September and tell me what went wrong in September."*

The assistant has produced the report below. **You are the analyst.** Meera will present whatever you sign off, so your job is to **verify, correct or reject** each AI insight using the data, and to say *why*.

**Ground rules (from the lecture):** every number must come from code you can inspect · an alert tells you where to look, a chart tells you what is true · the AI proposes, the human decides.

**How to work:** run the notebook top to bottom. Complete every cell marked `# TODO` (replace `None` with your code or answer) and write in the cells marked **✏️ Your answer**. Cells that depend on your TODO will tell you if something is still missing.

### The data: `freshkart_orders.csv` (synthetic)

| Column | Meaning |
|---|---|
| `order_id` | unique order reference |
| `order_date` | date of order |
| `city`, `store_id` | where the order was fulfilled |
| `zone` | `Urban` or `Suburban` delivery address |
| `delivery_partner` | `Partner A` or `Partner B` |
| `items`, `discount_pct`, `order_value` | basket size, discount applied, bill value (₹) |
| `distance_km`, `delivery_minutes`, `on_time` | delivery distance, time taken, 1 if ≤ 30 min |
| `rating` | customer rating 1–5 (blank if not given) |
| `app_sessions_7d`, `loyalty_points` | customer app activity and loyalty balance |

In [ ]:
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (6, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def ready(*vals):
    """Helper: tells you if a TODO above is still unanswered."""
    ok = all(v is not None for v in vals)
    if not ok:
        print("⚠ Complete the TODO cell(s) above first, then re-run this cell.")
    return ok

df = pd.read_csv("freshkart_orders.csv", parse_dates=["order_date"])
df.head()

## The AI assistant's report (to be validated)

| # | Insight type | AI statement |
|---|---|---|
| 1 | Trend | "Bengaluru orders grew steadily from January to September, by about **+56 orders per month**." |
| 2 | Distribution | "The **average order value is ₹465**." |
| 3 | Comparison | "Pune orders **more than doubled** from Q2 to Q3 (**+117%**), so the Pune expansion is a huge success." |
| 4 | Comparison | "Hyderabad orders **fell 28%** in September compared with August, **because delivery service there got worse**." |
| 5 | Comparison | "**Partner B is more punctual than Partner A** (93% vs 87% on time), so we should route all orders to Partner B." |
| 6 | Correlation | "Longer delivery times go with lower ratings (**r = −0.32** among rated orders)." |
| 7 | Correlation | "Bigger discounts and bigger baskets move together (**r = 0.38**)." |
| 8 | Comparison | "Suburban orders take about **5 minutes longer** to deliver than urban orders." |
| 9 | Prediction | "**Orders will recover in October.**" |

---
## Part 1 · Is the data trustworthy? Profile before you believe

An AI narrates a profile, so a flawed profile gives a confident but flawed story. Compute the basics yourself.

In [ ]:
profile = {
    "rows": len(df),
    "columns": df.shape[1],
    "duplicate_rows": int(df.duplicated().sum()),
    "rating_missing_pct": round(df["rating"].isna().mean() * 100, 1),
    "order_id_unique_pct": round(df["order_id"].nunique() / len(df) * 100, 1),
}
profile

In [ ]:
# From here on we work with the cleaned table (duplicates removed).
df = df.drop_duplicates().copy()

aov_mean = df["order_value"].mean()
aov_median = df["order_value"].median()
aov_skew = round(stats.skew(df["order_value"]), 1)

if ready(aov_mean, aov_median):
    print(f"mean = ₹{aov_mean:.0f} | median = ₹{aov_median:.0f} | skew = {aov_skew}")
    df["order_value"].plot.hist(bins=60, color="#4c78a8", title="Order value (₹)")
    plt.axvline(aov_mean, color="#d62728", label="mean"); plt.axvline(aov_median, color="black", label="median")
    plt.legend(); plt.show()

In [ ]:
# Where are the ratings missing?
df.groupby("city")["rating"].apply(lambda s: s.isna().mean() * 100).round(1)

**✏️ Your answer (Part 1):**
(a) Which AI insight(s) are affected by what you just found about the *average*, and how should the statement be rewritten?
(b) Insight 6 uses ratings. What risk does the missing-rating pattern create?

**(a)** Insight 2 is the claim affected directly: a mean alone can hide a skewed distribution and should not be reported as a typical order without the median and histogram. I would rewrite it as: “The mean order value is ₹(use the computed mean) and the median is ₹(use the computed median); report both because the distribution is skewed and the mean alone may not represent a typical order.”

**(b)** Missing ratings are not necessarily random. If some cities or types of customers rate less often, the correlation in insight 6 describes only the rated subset and may be biased; report the rating sample size and missingness by city.

---
## Part 2 · Defaults shape the story: bin width

The assistant looked at a 1-minute-bin histogram of one store's deliveries and wrote: *"Delivery times are **multimodal**, with three separate peaks."* Look at the same 150 orders with different bin widths.

In [ ]:
sample = df[df.store_id == "B3"].sample(150, random_state=1)["delivery_minutes"]
fig, axes = plt.subplots(1, 3, figsize=(11, 2.8))
for ax, w in zip(axes, [1, 4, 15]):
    ax.hist(sample, bins=np.arange(sample.min(), sample.max() + w, w), color="#4c78a8")
    ax.set_title(f"bin width = {w} min")
plt.tight_layout(); plt.show()

# A three-peak story is not robust if it changes with bin width.
multimodal_supported = False
# Four-minute bins provide a readable compromise without over-fragmenting the data.
best_bin_width = 4

**✏️ Your answer (Part 2):** In one sentence, what should the analyst do before accepting a shape claim from an automated report?

Change the bin width and axis scale, inspect the raw observations and compare the same sample across reasonable settings before calling a histogram multimodal.

---
## Part 3 · Trends, correlations and patterns

### 3a · Trend (insight 1)

In [ ]:
bengaluru = df[df.city == "Bengaluru"]
monthly = bengaluru.groupby(bengaluru["order_date"].dt.month).size()
months = monthly.index.values

slope = np.polyfit(months, monthly.values, 1)[0]
tau, p_trend = stats.kendalltau(months, monthly.values)

if ready(slope):
    print(f"slope = {slope:.1f} orders/month | Kendall tau = {tau:.2f}, p = {p_trend:.4f}")
    ax = monthly.plot(marker="o", color="#4c78a8", title="Bengaluru orders per month")
    ax.plot(months, np.polyval(np.polyfit(months, monthly.values, 1), months), "--", color="grey")
    plt.show()

### 3b · Correlation (insights 6 and 7)

In [ ]:
r_disc_items = df["discount_pct"].corr(df["items"], method="pearson")

# Same question, different units: minutes -> hours
rated = df.dropna(subset=["rating"]).copy()
rated["delivery_hours"] = rated["delivery_minutes"] / 60
r_minutes = rated["delivery_minutes"].corr(rated["rating"])
r_hours = rated["delivery_hours"].corr(rated["rating"])
r_unchanged = bool(np.isclose(r_minutes, r_hours))

if ready(r_disc_items, r_unchanged):
    print(f"discount~items r = {r_disc_items:.2f} | minutes~rating r = {r_minutes:.3f} | hours~rating r = {r_hours:.3f}")
    df.groupby("discount_pct")["items"].mean().plot(marker="o", color="#4c78a8",
        title="Average basket size by discount level (look at the data, not only r)")
    plt.show()

### 3c · Scanning many pairs

The assistant also scanned **every pair** of the 7 numeric columns below and flags any pair with p < 0.05 as a "finding".

In [ ]:
num_cols = ["order_value", "items", "discount_pct", "distance_km", "delivery_minutes", "app_sessions_7d", "loyalty_points"]

n_pairs = len(num_cols) * (len(num_cols) - 1) // 2
expected_false = n_pairs * 0.05
bonferroni = 0.05 / n_pairs

rows = []
for a, b in itertools.combinations(num_cols, 2):
    r, p = stats.pearsonr(df[a], df[b])
    rows.append((a, b, round(r, 3), p))
scan = pd.DataFrame(rows, columns=["x", "y", "r", "p"]).sort_values("p")

if ready(n_pairs, expected_false, bonferroni):
    print(f"pairs = {n_pairs} | expected false positives = {expected_false:.2f} | Bonferroni threshold = {bonferroni:.4f}")
    flagged = scan[scan["p"] < 0.05].copy()
    flagged["survives_bonferroni"] = flagged["p"] < bonferroni
    display(flagged)

**✏️ Your answer (3a–3c):**
(a) Insight 1 rests on 9 monthly points. Is the trend claim reasonable, and what would you show with it?
(b) Insight 7 says discounts and baskets "move together". Does the chart support the wording, and can the AI say discounts *cause* bigger baskets?
(c) Look at the flagged table. Which pair is most likely a false alarm, and what are **two** reasons you say so?

**(a)** A fitted slope summarises the average change, but nine monthly observations are too few to justify “grew steadily” without showing the month-by-month series and uncertainty; I would show the points, fitted line and Kendall trend result together.

**(b)** The correlation and grouped means can show an association, but the chart must be checked for a consistent pattern and outliers; neither proves that discounts cause larger baskets because customer mix, promotions and other factors may explain both.

**(c)** Treat the weakest, non-Bonferroni-surviving pair as the leading false-alarm candidate. Two reasons are that many pairs were tested (so some small p-values arise by chance) and the relationship is weak or lacks a plausible mechanism; statistical significance alone does not establish a useful finding.

### 3d · Patterns: grouping the stores

The assistant ran **PCA + k-means (k = 3)** on store-level features and proposed three store "segments".

In [ ]:
d = df.assign(month=df["order_date"].dt.month)
store = df.groupby("store_id").agg(
    orders=("order_id", "count"),
    median_value=("order_value", "median"),
    on_time=("on_time", "mean"),
    suburban_share=("zone", lambda s: (s == "Suburban").mean()),
    avg_discount=("discount_pct", "mean"),
    active_days=("order_date", "nunique"),
)
store["orders_per_active_day"] = store["orders"] / store["active_days"]
feats = store.drop(columns=["orders", "active_days"])

Z = StandardScaler().fit_transform(feats)
pcs = PCA(n_components=2).fit_transform(Z)
store["segment"] = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(Z)

fig, ax = plt.subplots(figsize=(5, 3.4))
ax.scatter(pcs[:, 0], pcs[:, 1], c=store["segment"], cmap="Set1", s=70)
for sid, (x, y) in zip(store.index, pcs):
    ax.annotate(sid, (x, y), xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2"); plt.show()
store[["segment", "orders_per_active_day", "on_time", "suburban_share"]].round(2).sort_values("segment")

**✏️ Your answer (3d):** The AI says these are three meaningful store "types". With only 10 stores, what is one reason a human should hesitate, and what would you check before naming the segments?

Ten stores are a very small sample for a three-cluster solution, so a few unusual stores or feature choices can move the cluster boundaries substantially. Before naming segments, inspect the original feature values and cluster profiles, test other values of k and random seeds, and check whether the groups are stable and operationally meaningful.

---
## Part 4 · From an observation to hypotheses (insight 4)

**Observation:** Hyderabad orders fell sharply in September. The AI jumped straight to "service got worse". A better assistant would list *competing* hypotheses and the evidence that separates them:

| | Hypothesis | Test |
|---|---|---|
| A | Delivery service got worse | on-time % in Aug vs Sep |
| B | A data problem: some store has missing rows | rows per store per week |
| C | Discounts were cut in September | mean discount, Aug vs Sep |
| D | Customers switched to a competitor | needs competitor/customer data we do not have |

In [ ]:
hyd = df[df.city == "Hyderabad"]
hyd_month = hyd.groupby(hyd["order_date"].dt.month).size()

change_pct = (hyd_month.loc[9] - hyd_month.loc[8]) / hyd_month.loc[8] * 100
first_test = "B"  # check store-week completeness first; a missing-data explanation is inexpensive to test

if ready(change_pct, first_test):
    print(f"Hyderabad Aug -> Sep: {change_pct:.1f}% | cheapest test first: {first_test}")

In [ ]:
# Run the tests for hypotheses B and A
weekly = hyd.groupby([pd.Grouper(key="order_date", freq="W"), "store_id"]).size().unstack(fill_value=0)
display(weekly.tail(9))

ontime = hyd.groupby(hyd["order_date"].dt.month)["on_time"].mean() * 100
ontime_change_pts = ontime.loc[9] - ontime.loc[8]
# Identify the Hyderabad store with a zero-order week in the displayed August/September window.
recent = weekly.tail(9)
zero_by_store = (recent == 0).any(axis=0)
culprit_store = zero_by_store[zero_by_store].index[0] if zero_by_store.any() else "No zero-order week"

if ready(ontime_change_pts, culprit_store):
    print(f"on-time change Aug->Sep = {ontime_change_pts:+.1f} points | store with the gap: {culprit_store}")

**✏️ Your answer (Part 4):** In two sentences, which hypotheses did the data support or rule out? Write the corrected version of insight 4 that you would be willing to show Meera.

The store-week table should be checked first: a gap in one Hyderabad store supports a data-completeness explanation, while the August-to-September on-time comparison tests whether delivery performance actually deteriorated; neither result alone proves customer switching. I would report: “Hyderabad orders changed by the percentage calculated above from August to September. The weekly store audit and the on-time percentage-point change should be reported alongside it; these checks do not establish that poorer delivery caused the decline.”

---
## Part 5 · Validate and correct two more claims

### 5a · "Pune orders more than doubled" (insight 3)

In [ ]:
pune = df[df.city == "Pune"].copy()
pune["quarter"] = pune["order_date"].dt.quarter
by_store = pune.pivot_table(index="store_id", columns="quarter", values="order_id", aggfunc="count", fill_value=0)
display(by_store)

old = by_store.loc[["P1", "P2"]]
like_for_like_pct = (old[3].sum() - old[2].sum()) / old[2].sum() * 100

if ready(like_for_like_pct):
    print(f"like-for-like growth = {like_for_like_pct:+.1f}%")

Now draw the *corrected* message. The chart below already greys the context and highlights the new stores (Knaflic: grey the context, one accent colour, title with the takeaway). **Write the title.**

In [ ]:
title = "Pune's Q3 increase includes new stores; compare P1 and P2 for like-for-like growth"

change = by_store[3] - by_store[2]
colors = ["#bdbdbd" if s in ("P1", "P2") else "#d62728" for s in by_store.index]
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(change.index, change.values, color=colors)
for x, v in zip(change.index, change.values):
    ax.text(x, v + 10, f"{v:+d}", ha="center")
ax.set_yticks([]); ax.spines["left"].set_visible(False)
ax.set_title(title, loc="left", fontsize=11, fontweight="bold")
plt.show()

### 5b · "Partner B is more punctual, send everything to B" (insight 5)

In [ ]:
overall = df.groupby("delivery_partner")["on_time"].mean().mul(100).round(1)
mix = (pd.crosstab(df["delivery_partner"], df["zone"], normalize="index") * 100).round(0)
print("Overall on-time %:\\n", overall, "\\n\\nShare of each partner's orders by zone (%):\\n", mix)

by_zone = (df.groupby(["zone", "delivery_partner"])["on_time"].mean().mul(100).unstack().round(1))
# Determine whether one partner wins in both zones, rather than comparing only the pooled averages.
if (by_zone["Partner A"] > by_zone["Partner B"]).all():
    better_in_every_zone = "Partner A"
elif (by_zone["Partner B"] > by_zone["Partner A"]).all():
    better_in_every_zone = "Partner B"
else:
    better_in_every_zone = "Neither consistently"
phenomenon = "Simpson's paradox"

if ready(by_zone, better_in_every_zone, phenomenon):
    display(by_zone)
    print(f"Better inside every zone: {better_in_every_zone} | effect: {phenomenon}")

**✏️ Your answer (Part 5):** (a) Rewrite insight 3 so it is honest. (b) Explain in plain language why Partner B *looks* better overall, and what question a careful human asks before accepting a comparison like insight 5.

**(a)** “Pune's total Q3 orders increased, but the headline growth includes orders from newly opened stores; among the two stores operating in both quarters (P1 and P2), like-for-like orders changed by [computed percentage].” This separates expansion from growth at existing stores.

**(b)** Partner B's pooled rate can look better because the partners handle different proportions of urban and suburban deliveries, and those zones have different baseline delivery difficulty. Compare the partners within each zone (and ideally control for distance, time and store); the key question is whether the advantage remains when like is compared with like.

---
## Part 6 · Triage the whole report

For each AI insight decide: **Accept** (correct as stated), **Correct** (the finding is useful but the wording or number must change) or **Reject** (not supported). Then measure the AI.

In [ ]:
decisions = {
    1: "Correct",   # trend direction may be useful, but the exact steady +56 claim needs qualification
    2: "Correct",   # report median and distribution alongside the mean
    3: "Correct",   # separate new-store growth from like-for-like growth
    4: "Reject",    # order decline does not establish service as the cause
    5: "Reject",    # pooled partner comparison is confounded by zone mix
    6: "Correct",   # association among rated orders; missing ratings limit generalisation
    7: "Correct",   # correlation is not causation
    8: "Accept",    # verify the group means in the output before presenting the size of the gap
    9: "Reject",    # a September trend cannot establish October recovery
}
analysts_true_insights = 7

accepted = sum(v == "Accept" for v in decisions.values())
if all(v in ("Accept", "Correct", "Reject") for v in decisions.values()):
    precision = accepted / len(decisions)
    recall = accepted / analysts_true_insights
    f1 = 2 * precision * recall / (precision + recall) if accepted else 0
    print(f"Accepted as stated: {accepted}/{len(decisions)} | precision = {precision:.2f} | recall = {recall:.2f} | F1 = {f1:.2f}")
    print(f"Review effort spent on insights that were not fully correct: {(len(decisions) - accepted) / len(decisions):.0%}")
else:
    print("⚠ Fill in all nine decisions with 'Accept', 'Correct' or 'Reject'.")

**✏️ Your answer (Part 6):** Using precision and recall, would you describe this AI as an *assistant* or a *replacement* for the analyst? Justify in two sentences.

I would call it an assistant, not a replacement: several claims require correction or rejection, and precision/recall show why blindly accepting the output would create avoidable reporting errors. A human analyst must verify the data, explain uncertainty and own the final decisions.

---
## Part 7 · Communicating, steering and validating

### 7a · Three kinds of reading
Each insight reads the data at a different level: **`data`** (restates what is plotted), **`between`** (relationships or patterns) or **`beyond`** (predictions and causal explanations; the risk zone).

In [ ]:
levels = {
    2: "data",
    6: "between",
    9: "beyond",
}
print(levels)

### 7b · Steering the assistant (interactive refinement)
Match each thing Meera types to the kind of feedback it is: **Reject**, **Constrain**, **Steer**, **Adjust** or **Pin**.

In [ ]:
user_actions = {
    '"Never use pie charts in this report"': "Constrain",
    '"Use 5-minute bins for delivery time"': "Adjust",
    '"Focus on on-time rate by city"': "Steer",
    '"Keep the like-for-like Pune chart on the dashboard"': "Pin",
    '"Not this chart again"': "Reject",
}
user_actions

### 7c · Which validation threat?
Meera now says: *"My real worry is why **repeat customers** are leaving. The AI answered a different question: how orders changed."* The AI's numbers were all computed correctly.

In [ ]:
threat = "wrong problem"
threat

**✏️ Your answer (Part 7):** Rewrite insight 2 *and* insight 4 as statements you would be comfortable presenting. Each rewrite should use the right measure or the right scope, avoid causal language the data cannot support, and be specific (no "interesting variation").

**Insight 2:** “The mean and median order values are reported together because the distribution is skewed; the median describes the typical order more robustly while the mean reflects the influence of high-value orders.”

**Insight 4:** “Hyderabad orders changed between August and September, and the store-week audit and on-time comparison provide checks on data completeness and service. These observations identify where to investigate but do not establish that poorer delivery caused the order change.”

---
## Reflection

Answer in 2–4 sentences each. There are no right-or-wrong answers here; we are looking for honest thinking about *your own* process.

**1. Automation bias.** Before you checked anything, which AI insight did you find most convincing? What made it convincing (the number, the tone, the chart, the confident wording), and would you have caught the problem without the checks?

**1.** The confident numerical trend claim initially sounded convincing because it gave a precise monthly growth figure and a clear business story. I might not have noticed the effect of new stores or month-to-month variation without plotting and checking the underlying data.

**2.** I would let the AI handle routine summaries, data type checks and draft charts when the steps are repeatable and easy to reverse. I would insist on human review for causal explanations, performance comparisons and predictions because those can influence staffing and business decisions.

**3.** For the Hyderabad decline, the tool should show order counts by store and week, missing periods, sample sizes and the August-to-September comparison in the same view. I would want to inspect the source rows, edit the analysis and undo or rerun the calculation with a different scope.

**4.** The analyst who signs off the report carries responsibility for checking it, while the vendor is responsible for clearly documenting the tool's limits and Meera is responsible for how the report is used. Next time I would keep a reproducible audit trail, verify every headline number and clearly label what the data cannot establish.

**2. Choosing the level of automation.** For FreshKart, name one task where you would let the AI work with little supervision and one where you would insist on a human check. What did you use to decide (stakes, reversibility, how visible errors are)?

**1.** The confident numerical trend claim initially sounded convincing because it gave a precise monthly growth figure and a clear business story. I might not have noticed the effect of new stores or month-to-month variation without plotting and checking the underlying data.

**2.** I would let the AI handle routine summaries, data type checks and draft charts when the steps are repeatable and easy to reverse. I would insist on human review for causal explanations, performance comparisons and predictions because those can influence staffing and business decisions.

**3.** For the Hyderabad decline, the tool should show order counts by store and week, missing periods, sample sizes and the August-to-September comparison in the same view. I would want to inspect the source rows, edit the analysis and undo or rerun the calculation with a different scope.

**4.** The analyst who signs off the report carries responsibility for checking it, while the vendor is responsible for clearly documenting the tool's limits and Meera is responsible for how the report is used. Next time I would keep a reproducible audit trail, verify every headline number and clearly label what the data cannot establish.

**3. Making trust cheaper.** Pick one problem you found (e.g. the Hyderabad gap, the missing ratings, or Partner B). What should the AI tool have *shown* (code, sample size, data gaps, uncertainty) so you could verify it faster, and what control would you want (edit, undo, switch off)?

**1.** The confident numerical trend claim initially sounded convincing because it gave a precise monthly growth figure and a clear business story. I might not have noticed the effect of new stores or month-to-month variation without plotting and checking the underlying data.

**2.** I would let the AI handle routine summaries, data type checks and draft charts when the steps are repeatable and easy to reverse. I would insist on human review for causal explanations, performance comparisons and predictions because those can influence staffing and business decisions.

**3.** For the Hyderabad decline, the tool should show order counts by store and week, missing periods, sample sizes and the August-to-September comparison in the same view. I would want to inspect the source rows, edit the analysis and undo or rerun the calculation with a different scope.

**4.** The analyst who signs off the report carries responsibility for checking it, while the vendor is responsible for clearly documenting the tool's limits and Meera is responsible for how the report is used. Next time I would keep a reproducible audit trail, verify every headline number and clearly label what the data cannot establish.

**4. Responsibility.** If Meera presents the corrected report and one number is still wrong, who is accountable, the AI, the tool's vendor, you or Meera? What would you do differently next time to carry that responsibility well?

**1.** The confident numerical trend claim initially sounded convincing because it gave a precise monthly growth figure and a clear business story. I might not have noticed the effect of new stores or month-to-month variation without plotting and checking the underlying data.

**2.** I would let the AI handle routine summaries, data type checks and draft charts when the steps are repeatable and easy to reverse. I would insist on human review for causal explanations, performance comparisons and predictions because those can influence staffing and business decisions.

**3.** For the Hyderabad decline, the tool should show order counts by store and week, missing periods, sample sizes and the August-to-September comparison in the same view. I would want to inspect the source rows, edit the analysis and undo or rerun the calculation with a different scope.

**4.** The analyst who signs off the report carries responsibility for checking it, while the vendor is responsible for clearly documenting the tool's limits and Meera is responsible for how the report is used. Next time I would keep a reproducible audit trail, verify every headline number and clearly label what the data cannot establish.